In [2008]:
import pandas as pd
import json
import os
import psycopg2

In [2009]:
path = "/Applications/python/nutristeppe/data/settings"
rows = []

for filename in os.listdir(path):
    if filename.endswith(".json"):
        bls_code = filename.replace(".json", "")
        
        with open(os.path.join(path, filename), "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if isinstance(data, list):
            data = data[0]
        
        per100 = data.get("nutrition_per_100g", {})
        
        ingredients = [ing["name"] for ing in data.get("ingredients", [])]
        
        rows.append({
            "bls_code":         bls_code,
            "protein":          per100.get("protein", {}).get("value", 0.0),
            "fat":              per100.get("fat", {}).get("value", 0.0),
            "carbohydrate":     per100.get("carbohydrate", {}).get("value", 0.0),
            "fiber":            per100.get("fiber", {}).get("value", 0.0),
            "sugar_mg":            per100.get("Сахар (общий)", {}).get("value", 0.0),
            "salt_total_mg":       per100.get("Поваренная соль всего", {}).get("value", 0.0),
            "saturated_fat_mg":    per100.get("Насыщенные жирные кислоты", {}).get("value", 0.0),
            "kilocalories":     data.get("calories_kcal_total_for_yield", 0.0),
            "ingredients":      ingredients,
            "techcard":         data.get("technology", ""),
        })

json_df = pd.DataFrame(rows)
json_df

,bls_code,protein,fat,carbohydrate,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,kilocalories,ingredients,techcard
0,Y363323,10.620,9.562,2.909,1.112,2.305,0.386,2.766,140.20,"[свиная рулька, морковь, лук репчатый, перец с...",Свиную рульку нарезают порционными кусками и о...
1,U501262,20.600,5.444,1.287,0.225,0.668,0.408,1.898,136.60,"[свинина, соль, перец черный, чеснок, паприка]","Свинину нарезать на порционные куски, натереть..."
2,Y221232,18.030,4.499,0.287,0.019,0.087,0.412,1.323,113.80,"[телятина, масло растительное, соль, перец чер...","Отбивную нарезать порционно, слегка отбить, по..."
3,U511262,21.490,3.952,0.002,0.001,0.002,0.408,0.883,121.60,"[Филе свинины нежирное, соль, перец черный, ма...","Филе свинины промыть, обсушить. Натрите солью ..."
4,16-041,11.710,1.840,0.125,0.042,0.122,0.296,0.419,63.88,"[филе камбалы, соль, перец черный, лимон, зелень]","Филе камбалы обсушить, равномерно распределить..."
...,...,...,...,...,...,...,...,...,...,...,...
2613,15-749,1.315,3.493,8.539,0.716,1.960,0.302,0.322,70.85,"[кабачок, картофель, лук репчатый, масло расти...","1. Лук репчатый, чеснок очистить и нарезать. К..."
2614,58160510,5.703,2.483,50.100,3.030,2.497,0.311,0.227,245.60,"[рис, морковь, зеленый горошек, масло растител...",Рис отваривают до готовности. Морковь нарезают...
2615,58164800,7.258,3.977,66.780,1.982,1.032,0.329,1.771,331.90,"[рис коричневый, Сливки 20% жирности, соль]","Рис промыть, затем отварить в подсоленной воде..."
2616,Y722243,7.319,6.620,1.970,0.322,1.970,0.491,1.922,96.73,"[яйцо, помидор, масло растительное, соль, пере...",Яйца разбивают и взбивают с солью и чёрным пер...


In [2010]:
weights = pd.read_csv("/Applications/python/nutristeppe/data/weight.csv", sep=';')
weights.rename(columns={"Порция": "serving_size_g", "code": "dish_category_code"}, inplace=True)
weights = weights[['dish_category_code', 'serving_size_g']]
weights

,dish_category_code,serving_size_g
0,GAR,150.0
1,GAR01,120.0
2,GAR0101,100.0
3,GAR0102,150.0
4,GAR0103,100.0
...,...,...
75,VEG02,250.0
76,BR,30.0
77,BR01,10.0
78,BR02,10.0


In [2011]:
df_new = pd.read_csv("/Applications/python/nutristeppe/data/type3_with_cat2.csv")
df_new

,bls_code,new_name,dish_category_code,type,type_5
0,B780200,Steiofebrot хлеб запеченный на каменных плитах,DGH0101,3,5
1,F201000,Абрикос,FR,3,1
2,F201100,Абрикос сырой,FR,3,5
3,F201101,Абрикос цельный,FR,3,5
4,F028700,Абрикосово-апельсиновый нектар,BEV04,3,4
...,...,...,...,...,...
5813,E108122,Яйцо пашот белок,EGG,2,5
5814,31105020,"Яйцо, жареное с маргарином",EGG,2,4
5815,17-743,Ячменная вода,BEV04,3,5
5816,51801010,Ячменный хлеб,DGH0102,3,2


In [2012]:
DB_CONFIG = {
    "host": "localhost",
    "database": "balaman",
    "user": "timurchiks",
    "password": "",
    "port": "5432"
}

conn = psycopg2.connect(**DB_CONFIG)
query = "select * from dishes;"
dishes = pd.read_sql(query, conn)
dishes

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_2285/3444354588.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dishes = pd.read_sql(query, conn)


,dish_id,bls_code,name,description,recipe_description,dish_category_id,dish_category_code,price,weight,kilocalories,...,carbohydrate,cfv,fvnl,index_health,type,type_5,new_name,cooking_technology,serving_method,quality_requirements
0,16564,F090400,Смешанные сушеные фрукты,None,None,43.0,DFR,0.0,25.0,287.0,...,0.00,70.0,5.0,3.50,3.0,5.0,Смешанные сушеные фрукты,None,None,None
1,16565,F090000,Смешанные фрукты,None,None,42.0,FR,0.0,125.0,85.0,...,0.00,30.0,5.0,4.00,3.0,5.0,Смешанные фрукты,None,None,None
2,16566,F090101,Смешанные фрукты сырые целые,None,None,42.0,FR,0.0,125.0,68.0,...,0.00,0.0,5.0,4.00,3.0,5.0,Смешанные свежие фрукты,None,None,None
3,19,X659012,"""Schupfudel"" (немецкая картофельная паста) (1)",None,None,5.0,GAR0201,0.0,200.0,100.0,...,0.00,0.0,100.0,3.75,2.0,2.0,Шупфнудель (немецкая картофельная паста),None,None,None
4,25,Y504322,"""Wildkrustel"" в панировке из дичи, обжаренная ...",None,None,7.0,MT01,0.0,300.0,141.0,...,0.00,0.0,0.0,2.75,2.0,2.0,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19320,9878,Y891312,Яблочные блины,None,None,16.0,DGH0201,0.0,100.0,194.9,...,25.46,0.0,30.0,2.25,2.0,1.0,Яблочные блины,"Муку смешивают с сахаром и солью, отдельно взб...","Подают горячими, можно посыпать сахаром или по...","Блины должны быть золотистого цвета, мягкие и ..."
19321,405,Y891433,Блинчики с творогом,None,None,38.0,CC,0.0,100.0,188.6,...,23.35,0.0,0.0,0.50,2.0,1.0,Блинчики с творогом,"В глубокой миске смешать муку, яйца, молоко и ...","Подают горячими, можно добавить свежие фрукты.","Блинчики должны быть золотистого цвета, мягкие..."
19322,899,Y892142,Сладкие картофельные клецки,None,None,3.0,GAR0103,0.0,100.0,178.5,...,37.54,26.3,26.3,4.00,2.0,1.0,Сладкие картофельные клецки,"Картофель отварить до полной готовности, воду ...","Подают горячими, можно посыпать сахаром или по...","Внешний вид: клецки одинаковой формы, без трещ..."
19323,3078,Y912130,Мясная котлета с картофельным салатом и горчицей,None,None,10.0,MT04,0.0,100.0,123.9,...,7.35,200.0,600.0,4.00,2.0,1.0,Мясная котлета с картофельным салатом и горчицей,Для приготовления котлеты говядину пропускаем ...,"Блюдо подают горячим, температура подачи 65-70°C.","Котлета должна быть золотистой, с хрустящей ко..."


In [2013]:
dishes = dishes[((dishes["type_5"] == 1) | (dishes["type_5"] == 2)) & ((dishes['type'] == 2) | (dishes['type'] == 3))]

In [2014]:
dishes['ingredients'] = None
dishes['techcard'] = None
dishes['fiber'] = None
dishes['sugar_mg'] = None
dishes['salt_total_mg'] = None
dishes['saturated_fat_mg'] = None
#sugar_mg, salt_total_mg, saturated_fat_mg

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_2285/3605098529.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dishes['ingredients'] = None
/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_2285/3605098529.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dishes['techcard'] = None
/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_2285/3605098529.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col

In [2015]:
dishes=dishes.drop(columns=['weight','image_url','name','dish_id','description','recipe_description','price','has_relation_with_products','health_factor','created_at','updated_at'])
dishes.rename(columns={"new_name": "name", "type_5": "availability_type", "cooking_technology": "steps"}, inplace=True)
df_new = df_new.rename(columns={"new_name": "name", "type_5": "availability_type"})

In [2016]:
dishes

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg
3,X659012,5.0,GAR0201,100.0,0.00,0.00,0.00,0.0,100.00,3.75,...,Шупфнудель (немецкая картофельная паста),None,None,None,None,None,None,None,None,None
4,Y504322,7.0,MT01,141.0,0.00,0.00,0.00,0.0,0.00,2.75,...,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None,None,None,None,None,None,None
6,X880522,6.0,GAR0202,115.0,0.00,0.00,0.00,16.7,30.00,2.25,...,Рис с горохом,None,None,None,None,None,None,None,None,None
7,X693213,41.0,EGG,131.0,0.00,0.00,0.00,0.0,33.33,2.25,...,Хоппель-Поппель (омлет с жареным картофелем и ...,None,None,None,None,None,None,None,None,None
8,X468713,29.0,CMT02,96.0,0.00,0.00,0.00,0.0,16.70,2.75,...,Гайсбургер Марш (картофель с говядиной),None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19320,Y891312,16.0,DGH0201,194.9,5.62,7.84,25.46,0.0,30.00,2.25,...,Яблочные блины,"Муку смешивают с сахаром и солью, отдельно взб...","Подают горячими, можно посыпать сахаром или по...","Блины должны быть золотистого цвета, мягкие и ...",None,None,None,None,None,None
19321,Y891433,38.0,CC,188.6,6.99,7.47,23.35,0.0,0.00,0.50,...,Блинчики с творогом,"В глубокой миске смешать муку, яйца, молоко и ...","Подают горячими, можно добавить свежие фрукты.","Блинчики должны быть золотистого цвета, мягкие...",None,None,None,None,None,None
19322,Y892142,3.0,GAR0103,178.5,5.28,0.81,37.54,26.3,26.30,4.00,...,Сладкие картофельные клецки,"Картофель отварить до полной готовности, воду ...","Подают горячими, можно посыпать сахаром или по...","Внешний вид: клецки одинаковой формы, без трещ...",None,None,None,None,None,None
19323,Y912130,10.0,MT04,123.9,10.48,5.85,7.35,200.0,600.00,4.00,...,Мясная котлета с картофельным салатом и горчицей,Для приготовления котлеты говядину пропускаем ...,"Блюдо подают горячим, температура подачи 65-70°C.","Котлета должна быть золотистой, с хрустящей ко...",None,None,None,None,None,None


In [2017]:
dishes['bls_code'] = dishes['bls_code'].astype(str).str.strip()
df_new['bls_code'] = df_new['bls_code'].astype(str).str.strip()

# Обновляем существующие строки
dishes_idx = dishes.set_index('bls_code')
df_new_idx = df_new.set_index('bls_code')
dishes_idx.update(df_new_idx)
dishes = dishes_idx.reset_index()

# Добавляем новые строки которых не было в dishes
new_codes = set(df_new['bls_code']) - set(dishes['bls_code'])
print(f"Новых кодов для добавления: {len(new_codes)}")  # проверяем сколько новых

new_rows = df_new[df_new['bls_code'].isin(new_codes)]
df_final = pd.concat([dishes, new_rows], ignore_index=True)

print(f"Строк до: {len(dishes)}, строк после: {len(df_final)}")
df_final

Новых кодов для добавления: 5291
Строк до: 6091, строк после: 11382


,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,Шупфнудель (немецкая картофельная паста),None,None,None,None,None,None,None,None,None
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None,None,None,None,None,None,None
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,Рис с горохом,None,None,None,None,None,None,None,None,None
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,Хоппель-Поппель (омлет с жареным картофелем и ...,None,None,None,None,None,None,None,None,None
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,Гайсбургер Марш (картофель с говядиной),None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11377,51119010,NaN,DGH0102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яичный хлеб Хала,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11378,E108122,NaN,EGG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яйцо пашот белок,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11379,31105020,NaN,EGG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"Яйцо, жареное с маргарином",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11380,17-743,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Ячменная вода,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2018]:
json_df = json_df.set_index("bls_code")
df_final = df_final.set_index("bls_code")

df_final.update(json_df)

df_final = df_final.reset_index()
json_df = json_df.reset_index()

In [2019]:
# Находим индексы строк, которые нужно удалить
# Условие: type == 1 И type_5 входит в список [3, 4, 5]
to_drop = df_final[(df_final['type'] == 1) | (df_final['availability_type'].isin([3, 4, 5]))].index
print(len(to_drop))
# Удаляем эти строки
df_final = df_final.drop(to_drop)
df_final

5208


,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,Шупфнудель (немецкая картофельная паста),None,None,None,None,None,None,None,None,None
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,"«Вильдкрустельн» из дичи в панировке, обжаренн...",None,None,None,None,None,None,None,None,None
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,Рис с горохом,None,None,None,None,None,None,None,None,None
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,Хоппель-Поппель (омлет с жареным картофелем и ...,None,None,None,None,None,None,None,None,None
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,Гайсбургер Марш (картофель с говядиной),None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11341,F015600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочно-морковный сок,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11342,Y840353,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочный компот,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11343,Y840343,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочный компот с изюмом,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11346,F115600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Яблочный сок,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2020]:
df_final = df_final.merge(weights, on="dish_category_code", how="left")
df_final

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g
0,X659012,5.0,GAR0201,100.0,0.0,0.0,0.0,0.0,100.00,3.75,...,None,None,None,None,None,None,None,None,None,100.0
1,Y504322,7.0,MT01,141.0,0.0,0.0,0.0,0.0,0.00,2.75,...,None,None,None,None,None,None,None,None,None,90.0
2,X880522,6.0,GAR0202,115.0,0.0,0.0,0.0,16.7,30.00,2.25,...,None,None,None,None,None,None,None,None,None,100.0
3,X693213,41.0,EGG,131.0,0.0,0.0,0.0,0.0,33.33,2.25,...,None,None,None,None,None,None,None,None,None,140.0
4,X468713,29.0,CMT02,96.0,0.0,0.0,0.0,0.0,16.70,2.75,...,None,None,None,None,None,None,None,None,None,150.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6169,F015600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0
6170,Y840353,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0
6171,Y840343,NaN,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0
6172,F115600,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0


In [2021]:
df_final[["protein", "fat", "carbohydrate"]] *= 1000

In [2022]:
df_final['kilocalories_portion'] = df_final['kilocalories'] * df_final['serving_size_g']/100

In [2023]:
# Создаем новую колонку 'Сумма_БЖУ'
df_final['calculated_kcal'] = ((df_final['protein'] + df_final['carbohydrate']) * 4 + (df_final['fat'] * 9))/1000
# Выбираем только интересующие нас данные
result_df = df_final[['bls_code', 'name','serving_size_g', 'dish_category_code', 'protein', 'fat', 'carbohydrate', 'calculated_kcal', 'kilocalories', 'kilocalories_portion']]
result_df['sub'] = result_df['kilocalories'] - result_df['calculated_kcal']
# ascending=False означает "по убыванию"
# result_df = result_df[(result_df["protein"] > 0) & (result_df["fat"] > 0) & (result_df["carbohydrate"] > 0)]
result_df = result_df.sort_values(by='sub', ascending=False)

result_df

/var/folders/1v/01d_s44n27b5nmlgynf1jxd40000gp/T/ipykernel_2285/2555114831.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['sub'] = result_df['kilocalories'] - result_df['calculated_kcal']


,bls_code,name,serving_size_g,dish_category_code,protein,fat,carbohydrate,calculated_kcal,kilocalories,kilocalories_portion,sub
961,H160600,Обжаренный орех пекан,30.0,SN02,0.0,0.0,0.0,0.0,716.0,214.8,716.0
1842,H140600,Гикори жареный,30.0,SN02,0.0,0.0,0.0,0.0,716.0,214.8,716.0
1963,H120162,Запечённые грецкие орехи,30.0,SN02,0.0,0.0,0.0,0.0,715.0,214.5,715.0
932,H190600,"Орех макадамия, обжаренный",30.0,SN02,0.0,0.0,0.0,0.0,706.0,211.8,706.0
365,H160700,Жареный и соленый пекан,30.0,SN02,0.0,0.0,0.0,0.0,702.0,210.6,702.0
...,...,...,...,...,...,...,...,...,...,...,...
6169,F015600,Яблочно-морковный сок,200.0,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6170,Y840353,Яблочный компот,10.0,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6171,Y840343,Яблочный компот с изюмом,10.0,BR02,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6172,F115600,Яблочный сок,200.0,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2024]:
df_final[df_final['name'] == 'Компот из голубой сливы'] #Y842612

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g,kilocalories_portion,calculated_kcal
5878,Y842612,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN


In [2025]:
df_final[df_final['bls_code'] == 'Y842612']

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,serving_size_g,kilocalories_portion,calculated_kcal
5878,Y842612,NaN,BEV04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0,NaN,NaN


In [2026]:
dishes[dishes['bls_code'] == 'Y842612']

,bls_code,dish_category_id,dish_category_code,kilocalories,protein,fat,carbohydrate,cfv,fvnl,index_health,...,name,steps,serving_method,quality_requirements,ingredients,techcard,fiber,sugar_mg,salt_total_mg,saturated_fat_mg


In [2027]:
df_new[df_new['bls_code'] == 'Y842612']

,bls_code,name,dish_category_code,type,availability_type
1502,Y842612,Компот из голубой сливы,BEV04,2,1


In [2028]:
json_df[json_df['bls_code'] == 'Y842612']

,bls_code,protein,fat,carbohydrate,fiber,sugar_mg,salt_total_mg,saturated_fat_mg,kilocalories,ingredients,techcard


In [2029]:
print(df_new[df_new['name'].str.contains("Компот", na=False)]['bls_code'].values)
print(df_final[df_final['name'].str.contains("Компот", na=False)]['bls_code'].values)

['F951900' 'Y841443' 'Y842412' 'Y846312' 'F949900' 'Y842343' 'F930200'
 'Y852312' 'Y844912' 'Y839312' 'Y840543' 'Y840563' 'F401222' 'F945900'
 'F948900' 'F947900' 'Y842512' 'Y842012' 'Y841312' 'Y842612' 'Y840843'
 'Y840863' 'Y840853' 'Y840953' 'Y840643' 'Y840653' 'Y840663' 'F944200'
 'Y842253' 'Y841363' 'Y841953' 'Y841112' 'Y841212' 'Y840753' 'Y840743'
 'Y840443' 'F943232' 'Y836312' 'Y837312' 'Y843312' 'F952900' 'F955900'
 'F952932' 'Y848212' 'Y840143' 'Y840153' 'Y840163' 'F957200' 'Y851512'
 'Y853512' 'F940200' 'Y849912' 'Y841543' 'Y841553' 'Y842443' 'Y841263'
 'Y888312' 'Y840253' 'Y840243' 'Y840263' 'F926222' 'F942900' 'F926232'
 'Y841643' 'F954900' 'F953900' 'F954932' 'Y865312' 'F941232' 'F927900'
 'F927200' 'Y840363']
['Y841443' 'Y842343' 'F930200' 'Y840543' 'Y842512' 'Y842612' 'Y840843'
 'Y840643' 'Y841212' 'Y840753' 'Y840443' 'Y837312' 'Y843312' 'Y848212'
 'F940200' 'Y841263' 'Y840253' 'Y841643' 'F953900' 'F927200']


In [2030]:
df_final['bls_code'].isna().sum()

np.int64(0)

In [2031]:
df_final.to_csv("30March.csv", index=False)